In [1]:
pip install pandas numpy matplotlib

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df= pd.read_csv("EQ Dataset.csv")
df.head()

,YEAR,MONTH,dp,dg,rf,re
0,1927,12,-3.143676,0.129874,0.053718,0.298791
1,1928,1,-3.142009,0.105342,0.042354,0.283000
2,1928,2,-3.117330,0.101963,0.048949,0.233187
3,1928,3,-3.203713,0.077818,0.043086,0.307920
4,1928,4,-3.242533,0.084212,0.042787,0.343425


In [6]:
# Inspect the data, how many years of data are available
print(df['YEAR'].nunique())
print(df['YEAR'].min(), df['YEAR'].max())

95
1927 2021


In [ ]:
# Step 4: total monthly return ret = rf + re
df['ret'] = df['rf'] + df['re']

# Step 5: check for missing values
print(df.isna().sum())

In [ ]:
# Step 7-8: mean log dividend-price ratio and persistence parameter kappa
bar_dp = df['dp'].mean()
kappa = 1 / (1 + np.exp(bar_dp))
print('bar_dp =', bar_dp)
print('kappa =', kappa)

In [ ]:
# Step 9: Campbell-Shiller present-value decomposition regressions for horizons H = 1,...,20 years
#
# Convention used (monthly steps, h = 1,...,12H):
#   future_return_m = sum_{h=1}^{12H} kappa^(h-1) * ret_{m+h}
#   change_d_m      = sum_{h=1}^{12H} kappa^(h-1) * (-dg_{m+h})
#   dp_H_m          = kappa^(12H) * dp_{m+12H}
# Rows m without a full 12H-month future window are left as NaN and dropped before each regression.

n = len(df)
ret_vals = df['ret'].to_numpy()
dg_vals = df['dg'].to_numpy()
dp_vals = df['dp'].to_numpy()

b_re_list = []
b_dg_list = []
b_dpH_list = []

for H in range(1, 21):
    horizon = 12 * H
    weights = kappa ** np.arange(horizon)  # kappa^(h-1) for h = 1, ..., horizon

    future_return = np.full(n, np.nan)
    change_d = np.full(n, np.nan)
    dp_H = np.full(n, np.nan)

    for m in range(n - horizon):
        future_return[m] = np.dot(weights, ret_vals[m + 1: m + 1 + horizon])
        change_d[m] = np.dot(weights, -dg_vals[m + 1: m + 1 + horizon])
        dp_H[m] = (kappa ** horizon) * dp_vals[m + horizon]

    df_H = pd.DataFrame({
        'dp': dp_vals,
        'future_return': future_return,
        'change_d': change_d,
        'dp_H': dp_H,
    }).dropna(subset=['future_return', 'change_d', 'dp_H'])

    b_re = np.polyfit(df_H['dp'], df_H['future_return'], 1)[0]
    b_dg = np.polyfit(df_H['dp'], df_H['change_d'], 1)[0]
    b_dpH = np.polyfit(df_H['dp'], df_H['dp_H'], 1)[0]

    b_re_list.append(b_re)
    b_dg_list.append(b_dg)
    b_dpH_list.append(b_dpH)

In [ ]:
# Step 13: plot the three coefficient series against the horizon H
horizons = list(range(1, 21))

plt.figure(figsize=(8, 5))
plt.plot(horizons, b_re_list, color='red', label='b_re (future return)')
plt.plot(horizons, b_dg_list, color='blue', label='b_dg (dividend growth)')
plt.plot(horizons, b_dpH_list, color='green', label='b_dpH (future dp)')
plt.xlabel('Horizon H (years)')
plt.ylabel('OLS slope coefficient on dp')
plt.title('Campbell-Shiller Decomposition Regression Coefficients vs. Horizon')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()